In [ ]:
# Install the libraries we need (only needed the first time)
# pypdf  -> reads text from PDF files
# pandas -> shows the results as a table
%pip install pypdf pandas

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json                  # to save the result as a JSON file
import re                    # regular expressions: to find and replace text patterns
import pandas as pd          # to show the modules as a table
from pypdf import PdfReader  # to open and read the PDF

In [ ]:
# Open the PDF (it must be in the same folder as this notebook)
reader = PdfReader("modulhandbuch.pdf")

# Take the text out of every page and put it in a list
# "or ''" means: if a page has no text, use an empty string instead
raw_pages = [page.extract_text() or "" for page in reader.pages]

print(len(raw_pages), "pages")   # should print 30

# Show page 4 the raw way, so we can see the messy parts
# repr() also shows hidden characters, like \n for a line break
print(repr(raw_pages[3][:800]))



In [2]:
# Open the PDF (it must be in the same folder as this notebook)
reader = PdfReader("modulhandbuch.pdf")

# Take the text out of every page and put it in a list
# "or ''" means: if a page has no text, use an empty string instead
raw_pages = [page.extract_text() or "" for page in reader.pages]

print(len(raw_pages), "pages")   # should print 30

# Show page 4 the raw way, so we can see the messy parts
# repr() also shows hidden characters, like \n for a line break
print(repr(raw_pages[3][:800]))

30 pages
'Modulhandbuch für den Bachelor-Studiengang  HAW Hamburg \nWirtschaftsinformatik  Department Informatik \n4 \n \nModulbezeichnung Grundlagen der Mathematik Kürzel GM / GMP \nLehrveranstaltung(en) SeU:  Grundlagen der Mathematik \nPraktikum:  Grundlagen der Mathematik \nFach-\nsemester \n1 \nArbeitsaufwand 54 Std. SeU, 18 Std. Praktikum, 108 Std. Eigenarbeit/Selbststudium Dauer ein Semester \nModulverantwortliche(r) Prof. Dr. Julia Padberg Turnus jährlich \nArt des Moduls Pflichtmodul CP 6 \nVoraussetzungen keine SWS 3+1  \nVerwendbarkeit für den Studiengang „Wirtschaftsinformatik“ Sprache deutsch \nZu erwerbende \nKompetenzen / \nLernergebnisse \nDie Studierenden \n\uf0b7 können wichtige mathematische Strukturen sicher verwenden \n\uf0b7 können formal mathematisch arbeiten \n\uf0b7 können Beweistechniken anwenden \n\uf0b7 können fü'


In [3]:
# The same header is printed at the top of every page.
# This pattern matches it so we can delete it.
# \s+ means "one or more spaces or line breaks", \d* means "a page number, if there is one"
PAGE_HEADER = re.compile(
    r"Modulhandbuch für den Bachelor-Studiengang\s+HAW Hamburg\s*"
    r"Wirtschaftsinformatik\s+Department Informatik\s*\d*"
)

def clean(text):
    """Fix the problems that come from reading the PDF."""
    text = text.replace("\uf0b7", "-")                      # strange bullet symbol -> dash
    text = text.replace("\u2022", "-")                      # normal bullet symbol -> dash
    text = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", "", text)    # delete invisible control characters
    text = re.sub(r"[ \t]+", " ", text)                     # many spaces in a row -> one space
    text = re.sub(r" *\n *", "\n", text)                    # delete spaces before/after line breaks
    text = text.replace("Fach-\nsemester", "Fachsemester")  # join the word split over two lines
    text = PAGE_HEADER.sub("", text)                        # delete the page header
    text = re.sub(r"\n{3,}", "\n\n", text)                  # max one empty line in a row
    return text.strip()                                     # remove spaces at the start and end

# Clean every page
pages = [clean(t) for t in raw_pages]

# Show page 4 again, now cleaned. Compare it with the output of Cell 3.
print(pages[3][:800])


Modulbezeichnung Grundlagen der Mathematik Kürzel GM / GMP
Lehrveranstaltung(en) SeU: Grundlagen der Mathematik
Praktikum: Grundlagen der Mathematik
Fachsemester
1
Arbeitsaufwand 54 Std. SeU, 18 Std. Praktikum, 108 Std. Eigenarbeit/Selbststudium Dauer ein Semester
Modulverantwortliche(r) Prof. Dr. Julia Padberg Turnus jährlich
Art des Moduls Pflichtmodul CP 6
Voraussetzungen keine SWS 3+1
Verwendbarkeit für den Studiengang „Wirtschaftsinformatik“ Sprache deutsch
Zu erwerbende
Kompetenzen /
Lernergebnisse
Die Studierenden
- können wichtige mathematische Strukturen sicher verwenden
- können formal mathematisch arbeiten
- können Beweistechniken anwenden
- können für die Wirtschaftsinformatik nötige Elemente der Boolschen Algebra anwenden
- können Graphen in Maximalflussproblemen anwenden
Inha


In [4]:
# Every module is on its own page, and that page starts with "Modulbezeichnung".
# So we keep only those pages, and each page becomes one chunk.
# Other pages (title page, overview, exam rules) are skipped.
chunks = [
    {"page": i + 1, "text": text}             # i + 1 because Python counts from 0, pages from 1
    for i, text in enumerate(pages)           # enumerate gives us the number AND the text
    if text.startswith("Modulbezeichnung")
]

print(len(chunks), "modules")   # should print 27

27 modules


In [10]:
assert len(chunks), 27

In [5]:
def find(pattern, text):
    """Search for a pattern and return the part inside ( ). Return None if not found."""
    match = re.search(pattern, text, re.DOTALL)        # DOTALL: the match can go over line breaks
    if match:
        return " ".join(match.group(1).split())        # tidy up: join lines, single spaces
    return None

def parse_module(chunk):
    """Turn one module chunk into a dictionary with the main fields."""
    text = chunk["text"]
    return {
        "id": f"module_{chunk['page']:02d}",     # a unique id, e.g. module_04
        "page": chunk["page"],                   # page number, so we can cite the source later
        # the name is whatever is between "Modulbezeichnung" and "Kürzel"
        "name": find(r"Modulbezeichnung\s+(.+?)\s+Kürzel", text),
        # the Kürzel is the rest of that line
        "kuerzel": find(r"Kürzel\s+(.+?)\n", text),
        # the semester number after "Fachsemester" or "Semester"
        "semester": find(r"(?:Fachsemester|\bSemester)\s*(\d[\d\-– und]*?)\s*\n", text),
        # the person between "Modulverantwortliche(r)" and "Turnus"
        "verantwortlich": find(r"Modulverantwortliche\(r\)\s+(.+?)\s+Turnus", text),
        # the first word after "Art des Moduls", e.g. Pflichtmodul
        "art": find(r"Art des Moduls\s+(\w+)", text),
        # the number after "CP"
        "cp": find(r"CP\s+([\d\s\+]+?)\s*\n", text),
        # the rest of the line after "Sprache" (or until the end of the text)
        "sprache": find(r"Sprache\s+(.+?)(?:\n|$)", text),
        # the full cleaned text, used later for the search
        "text": text,
    }

# Do this for all 27 chunks
modules = [parse_module(c) for c in chunks]

# Put it in a table to check it
df = pd.DataFrame(modules)

# Count empty fields per column. All should be 0.
print(df.isna().sum())

# Show the main columns as a table
df[["page", "name", "kuerzel", "semester", "cp", "verantwortlich"]]


id                0
page              0
name              0
kuerzel           0
semester          0
verantwortlich    0
art               0
cp                0
sprache           0
text              0
dtype: int64


,page,name,kuerzel,semester,cp,verantwortlich
0,4,Grundlagen der Mathematik,GM / GMP,1,6,Prof. Dr. Julia Padberg
1,5,Grundlagen der Wirtschaftsinformatik,GWI / GWIP,1,6,Prof. Dr. Martin Schultz
2,6,Programmiermethodik I,PM1,1,6,Prof. Dr. Philipp Jenke
3,7,Programmiertechnik,PT / PTP,1,6,Prof. Dr. Philipp Jenke
4,8,Betriebswirtschaft I,BWL1 / BWLP1,1,6,Prof. Dr. Jens-Eric von Düsterlho
5,9,Quantitative Methoden,QM / QMP,2,6,Prof. Dr. Marina Tropmann-Frick
6,10,Theoretische Informatik,TH / THP,2,6,Prof. Dr. Michael Köhler-Bußmeier
7,11,Informationssysteme I,IN1 / INP1,2,6,Prof. Dr. Olaf Zukunft
8,12,Programmiermethodik II,PM2 / PMP2,2,6,Prof. Dr. Philipp Jenke
9,13,Betriebswirtschaft II,BWL2 / BWLP2,2,6,Prof. Dr. Jens-Eric von Düsterlho


In [6]:
# Save all modules in one JSON file
# ensure_ascii=False keeps ä, ö, ü readable
# indent=2 makes the file easy to read
with open("modules.json", "w", encoding="utf-8") as f:
    json.dump(modules, f, ensure_ascii=False, indent=2)

print("Saved", len(modules), "modules to modules.json")

Saved 27 modules to modules.json
